<a href="https://colab.research.google.com/github/letiBri/MaskArchitectureAnomaly_CourseProject/blob/main/STEP8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anomaly Segmentation Eomt Coco e Cityscapes (Step 8)
# io metterei in un altro file l'evauation su coco fine tuned per evitare casini

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eval')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')


In [4]:
!pip install ood-metrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14

In [9]:
!pip install --upgrade torch torchvision torchaudio lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/

In [4]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
#from erfnet import ERFNet
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

In [5]:
!python3 -m pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 59.2 MB/s e

# Setup

In [5]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

device = 0  # change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"  # drive folder of the cityscapes val set per trovare la mIoU

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)

with open("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml", "r") as f:
    config_coco = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image


# Load Dataset


In [6]:
data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
)
data.setup()

# Load model

In [25]:
# Scegliamo il config e il file dei pesi in base alla scelta sopra
use_coco = True # Metto True per il modello COCO, False per quello Cityscapes

# Scegliamo il config e il file dei pesi in base alla scelta sopra
if use_coco:
    current_config = config_coco
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
    target_img_size = (640, 640)
    num_classes_to_load = 133
else:
    current_config = config_cs
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"
    num_classes_to_load = data.num_classes # Usa le classi del dataset (solitamente 19 o 20)
    target_img_size = data.img_size

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_to_load,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

# Load Pretrained

In [26]:
if use_coco:
    target_img_size = (640, 640)
else:
    # Prova a prenderlo dal config, se non c'è usa quello del dataset (1024)
    target_img_size = data.img_size

model_kwargs_final = {k: v for k, v in model_kwargs.items()}

# FIX CRUCIALE: Rimuoviamo eventuali 'num_classes' dai kwargs che potrebbero sovrascrivere il nostro valore
if "num_classes" in model_kwargs_final:
  del model_kwargs_final["num_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs_final,
    )
    .eval()
    .to(device)
)

if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True
    )
    model.load_state_dict(state_dict, strict=False)

# Semantic Inference

In [9]:
IGNORE_INDEX = 255

# DEFINIZIONE MAPPATURA UNIFICATA (COCO -> CITYSCAPES TRAIN_IDS)
# Questo dizionario mappa i category_id di COCO nei trainId di Cityscapes (0-18)
coco_to_cityscapes_map = {
    # --- THINGS (Istanze) ---
    0: 11,   # person -> person
    1: 18,   # bicycle -> bicycle
    2: 13,   # car -> car
    3: 17,   # motorcycle -> motorcycle
    5: 15,   # bus -> bus
    6: 16,   # train -> train
    7: 14,   # truck -> truck
    9: 6,    # traffic light -> traffic light
    11: 7,   # stop sign -> traffic sign (generico)

    # --- STUFF (Sfondi/Regioni) ---
    # Nota: Assicurati che questi ID corrispondano al file JSON di COCO Stuff fornito
    100: 0,  # road -> road
    123: 1,  # pavement-merged -> sidewalk
    129: 2,  # building-other-merged -> building
    91: 2,   # house -> building
    131: 3,  # wall-other-merged -> wall
    117: 4,  # fence-merged -> fence
    116: 8,  # tree-merged -> vegetation
    125: 8,  # grass-merged -> vegetation
    90: 9,   # gravel -> terrain
    102: 9,  # sand -> terrain
    119: 10, # sky-other-merged -> sky
}

def translate_preds(pred_array):
    """
    Converte i label predetti da COCO (0-132) nei label Cityscapes (0-18).
    Tutto ciò che non è nel mapping diventa IGNORE_INDEX (255).
    """
    translated = np.full_like(pred_array, fill_value=IGNORE_INDEX)
    for coco_id, city_id in coco_to_cityscapes_map.items():
        translated[pred_array == coco_id] = city_id
    return translated

def infer_semantic(img, target):
    model.eval()

    # Forza il modello a usare la dimensione della finestra corretta
    # (640 per COCO, 1024 per Cityscapes)
    model.window_size = target_img_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()

    # Se il modello caricato è COCO (134 classi), traduciamo i risultati
    if use_coco: # se siamo nel modello coco
        pred_array = translate_preds(pred_array)

    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    return pred_array, target_array

In [10]:
# prediction cityscapes
img_cs, target_cs = data.val_dataloader().dataset[img_idx]
pred_array_cs, target_array_cs = infer_semantic(img_cs, target_cs)

In [27]:
# prediction coco
img_cc, target_cc = data.val_dataloader().dataset[img_idx]
pred_array_cc, target_array_cc = infer_semantic(img_cc, target_cc)

# Evaluation mIoU

In [11]:
import numpy as np
from tqdm import tqdm

def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))

# setting
num_eval_classes = 19
hist = np.zeros((num_eval_classes, num_eval_classes))
val_loader = data.val_dataloader()

print(f"evaluation on {len(val_loader)} images...")
print(f"Current model: {'COCO' if use_coco else 'Cityscapes'}")

# evaluation
for i, batch in enumerate(tqdm(val_loader)):
    img = batch[0][0]
    target = batch[1][0]

    pred_array, target_array = infer_semantic(img, target)

    hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# calcolo finale
ious = per_class_iu(hist)
miou = np.nanmean(ious)

# visualization results
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"Results MIOU - {'COCO' if use_coco else 'CITYSCAPES'}")
print("="*30)
for name, iou in zip(classes, ious):
    print(f"{name:15}: {iou*100:6.2f}%")
print("-" * 30)
print(f"MEAN IoU: {miou*100:6.2f}%")
print("="*30)

evaluation on 500 images...
Current model: Cityscapes


100%|██████████| 500/500 [06:09<00:00,  1.35it/s]


Results MIOU - CITYSCAPES
road           :  98.40%
sidewalk       :  87.36%
building       :  94.15%
wall           :  66.07%
fence          :  65.49%
pole           :  71.04%
traffic light  :  75.00%
traffic sign   :  82.13%
vegetation     :  93.02%
terrain        :  66.60%
sky            :  95.54%
person         :  85.38%
rider          :  71.11%
car            :  95.54%
truck          :  81.79%
bus            :  90.32%
train          :  77.36%
motorcycle     :  74.35%
bicycle        :  81.27%
------------------------------
MEAN IoU:  81.68%


# Evaluation anomaly metrics

In [12]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)



In [28]:
parser = ArgumentParser()

parser.add_argument(
    "--input",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg", # change the default path based on the dataset folder
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png"
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg"
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FD_LostFound_full/images/*.png"
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.png"
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)

parser.add_argument('--num-workers', type=int, default=2)
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
parser.add_argument('--device', type=str, default='cuda', help="cpu or cuda, the device used for evaluation")
args = parser.parse_args(args=[])


anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []


In [14]:
### NEW

def get_eomt_logits(model, img_tensor, target_size):
    """
    Prende l'immagine, la passa in EoMT usando la logica a finestre
    e restituisce i logits semantici pixel-per-pixel [1, C, H, W].
    """
    model.eval()
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        # La rete si aspetta una lista di tensori
        imgs = [img_tensor.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        # Gestione dei crop (specifico di EoMT) questi metodi fanno parte dell'architettura EOMT
        crops, origins = model.window_imgs_semantic(imgs)

        # Forward pass vero e proprio
        mask_logits_per_layer, class_logits_per_layer = model(crops)

        # Upsampling delle maschere
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_size, mode="bilinear"
        )

        # Fusione di maschere e classi per ottenere i logits dei pixel questi metodi fanno parte dell'architettura EOMT
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )

        # Ricostruzione dell'immagine intera dai crop questi metodi fanno parte dell'architettura EOMT
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)

    return logits # Forma: [1, NUM_CLASSI, H, W]

In [15]:
# ## RbA
# if args.device == 'cuda' and (not torch.cuda.is_available()):
#     print("Warning: Cuda is requested but cuda is not available. CPU will be used.")
#     args.device = 'cpu'
# DEVICE = torch.device(args.device)

# def get_RbA(model, x, **kwargs):

#     with torch.no_grad():
#         out = model([{"image": x[0].to(DEVICE)}])

#     logits = out[0]['sem_seg']

#     return -logits.tanh().sum(dim=0)
# # ##

# # Funzione RbA NEW
# def get_RbA(model, img_tensor):
#     model.eval()
#     with torch.no_grad():
#         # RbA solitamente lavora sull'output standard del predictor/model
#         # Passiamo l'immagine come lista di dizionari (formato Detectron2 richiesto da EoMT)
#         inputs = [{"image": img_tensor.to(device)}]
#         outputs = model(inputs)

#         # Estraiamo i logits semantici [C, H, W]
#         # RbA calcola l'energia negativa tramite la tangente iperbolica
#         logits = outputs[0]['sem_seg']
#         rba_score = -torch.tanh(logits).sum(dim=0)

#     return rba_score.cpu().numpy()

In [29]:
# Controlliamo se args.input è una lista o una stringa
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

# new # Creiamo una cartella per salvare i logits (Cruciale per la Temperature Scaling!)
os.makedirs("saved_logits", exist_ok=True)

for path in glob.glob(os.path.expanduser(str(input_pattern))):

    img_pil = Image.open(path).convert('RGB').resize((1024,512), Image.BILINEAR) # sarebbe input_transform() senza toTensor()
    images = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1)
    #images = input_transform((Image.open(path).convert('RGB'))).float() #.unsqueeze(0).float().cuda()

    # Otteniamo i logits dal modello EoMT che restituisce una lista
    logits_list = get_eomt_logits(model, images, target_img_size) # NEW
    logits = logits_list[0].unsqueeze(0) # Estraiamo il tensore dalla lista e gli aggiungiamo la dimensione batch
    # così diventa [1, 19, Altezza, Larghezza]

    # SALVATAGGIO LOGITS SU DISCO (PRO TIP)
    # Li salviamo sulla CPU per non riempire la RAM della scheda video
    nome_file_logit = os.path.basename(path).replace(".jpg", ".pt").replace(".png", ".pt").replace(".webp", ".pt")
    torch.save(logits.cpu(), os.path.join("saved_logits", nome_file_logit))

    #compute max logit
    #anomaly_result_maxlogit = 1.0 - np.max(logits.squeeze(0).data.cpu().numpy(), axis=0) # forse meglio fare il max con pytorch
    anomaly_result_maxlogit = 1.0 - torch.max(logits.squeeze(0), dim=0)[0].cpu().numpy() # più efficiente

    #compute MSP
    probs = torch.nn.functional.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    #compute Max Entropy
    probs = probs.squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    #compute RbA
    anomaly_result_rba = -torch.sum(torch.tanh(logits.squeeze(0)), dim=0).cpu().numpy() #get_RbA(model, images)

    # gestione maschere
    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
      #in RoadAnomaly 2 indica anomalia --> viene trasformato in 1 per uniformare
        ood_gts = np.where((ood_gts==2), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue   #se non c'è nessuna anomalia, allora salta l'immagine e passa a valutare la successiva
    else:
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)
          anomaly_score_list_rba.append(anomaly_result_rba)

    # per evitare out of memory error
    del logits, anomaly_result_maxlogit,anomaly_result_msp, anomaly_result_maxentropy, anomaly_result_rba, ood_gts, mask
    torch.cuda.empty_cache()


In [30]:
ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)
anomaly_scores_rba = np.array(anomaly_score_list_rba)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')


#MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')


#max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')


#rba
ood_out = anomaly_scores_rba[ood_mask]
ind_out = anomaly_scores_rba[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score rba: {prc_auc*100.0}')
print(f'FPR@TPR95 rba: {fpr*100.0}')


AUPRC score maxlogit: 18.81068457462913
FPR@TPR95 maxlogit: 96.92310047245239
AUPRC score msp: 18.570578756538612
FPR@TPR95 msp: 97.0020931869476
AUPRC score maxentropy: 21.08539261864063
FPR@TPR95 maxentropy: 96.36013654986589
AUPRC score rba: 15.72625165268697
FPR@TPR95 rba: 88.30116170757275


In [22]:
anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []

# per svuotare le liste nella stessa sessione quando voglio usare cityscapes oppure rieseguo la cella dove le creo